# RoPE

In [2]:
import torch
import torch.nn as nn
import numpy as np
import math

In [5]:
class rope(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.hidden_dim = hidden_dim
        
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)

    def rotate_half(self, x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim = -1).flatten(-2)

    def apply_rope(self, x):
        seq_len = x.shape[1]
        inv_freq = 1.0 / (10000 ** (torch.arange(0, self.hidden_dim, 2, device = x.device) / self.hidden_dim))
        positions = torch.arange(seq_len, device = x.device)
        angles = (positions[:, None] * inv_freq[None, :])
        sin = torch.sin(angles)
        cos = torch.cos(angles)
        sin = sin.repeat_interleave(2, dim = -1)
        cos = cos.repeat_interleave(2, dim = -1)
        sin = sin.unsqueeze(0)
        cos = cos.unsqueeze(0)
        x = (x * cos + self.rotate_half(x) * sin)
        return x
    
    def forward(self, x):
        seq_len = x.shape[1]
        
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        pos = torch.arange(0, seq_len)
        freq = 1 / (10000 ** (torch.arange(0, self.hidden_dim, 2) / self.hidden_dim))
        angles = pos[:, None] * freq[None, :]
        sin = torch.sin(angles)
        cos = torch.cos(angles)

        
        
        scores = Q @ K.transpose(-2,-1)
        wts = scores / math.sqrt(K.shape[-1])
        wts = torch.softmax(wts, dim = -1)
        output = wts @ V
        return output

# KV Caching

In [18]:
class kvc(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)
        
        self.K_cache = None
        self.V_cache = None

    def reset_cache(self):
        self.K_cache = None
        self.V_cache = None

    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        if self.K_cache is None:
            self.K_cache = K
            self.V_cache = V

        else:
            self.K_cache = torch.cat([self.K_cache, K], dim = 1)
            self.V_cache = torch.cat([self.V_cache, V], dim = 1)

        scores = (Q @ self.K_cache.transpose(-2,-1))
        scores = scores / math.sqrt(Q.shape[-1])
        wts = torch.softmax(scores, dim = -1)
        output = wts @ self.V_cache
        return output

In [19]:
model = kvc(128)
model.reset_cache()

In [20]:
x1 = torch.randn(1,1,128)
out1 = model(x1)
out1

tensor([[[ 0.4767, -0.4599, -0.2590, -0.1893,  0.1012,  0.5956, -0.4844,
           1.0754, -0.2162, -0.2889,  0.0118,  0.2297,  0.3344,  0.2267,
           0.3199, -0.2667,  0.4986, -0.1503,  0.3882,  0.9865, -0.2945,
          -0.7085,  0.8087,  0.3219,  0.5283, -0.6705,  0.6574,  0.2747,
           0.1434,  0.1674,  0.9523,  0.2468, -0.1930, -1.0488,  1.1881,
          -0.2522,  0.7645, -1.8834,  0.1805, -0.7335,  0.1994,  0.3881,
          -0.2088,  0.4623, -0.0831,  0.9227, -0.1629, -0.7111,  1.2576,
           0.0605,  0.4670,  0.5134,  0.3296, -1.7255,  0.7620, -1.3310,
           0.9453,  0.9567,  0.5482,  0.4058, -0.5753,  0.9600, -0.2663,
          -0.1077, -0.3248, -0.7257,  0.8783,  0.1460,  1.3913, -0.6067,
           0.4934,  0.3600, -0.2371, -0.2536, -0.3560,  0.6776, -0.1153,
          -0.2513,  0.4139,  0.3204,  0.0855, -1.1593, -0.1583,  1.0745,
          -0.4790, -0.1374,  0.6866, -1.1817,  0.1480, -0.2094,  0.7328,
          -0.0476,  0.0702, -0.7283, -0.5628,  0.32

# Multi-Query Attention

In [26]:
class mqa(nn.Module):
    def __init__(self, hidden_dim, n_heads):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_heads = n_heads

        self.head_dim = int(hidden_dim / n_heads)

        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, self.head_dim)
        self.Wv = nn.Linear(hidden_dim, self.head_dim)
        self.Wo = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(batch_size, seq_len, self.n_heads, -1).transpose(1,2)
        K = K.view(batch_size, seq_len, 1, -1).transpose(1,2)
        V = V.view(batch_size, seq_len, 1, -1).transpose(1,2)

        scores = Q @ K.transpose(-2, -1)
        scores = scores // math.sqrt(self.head_dim)

        weights = torch.softmax(scores, dim = -1)
        output = weights @ V
        output = output.transpose(1, 2).reshape(batch_size, seq_len, hidden_dim)
        output = self.Wo(output)

        return output

In [31]:
model = mqa(128, 8)
x = torch.randn(4, 20, 128)
model(x)

tensor([[[ 1.6029e-01, -1.2163e-01,  1.0972e-02,  ..., -1.9944e-04,
          -4.4748e-03, -3.1309e-02],
         [ 1.4365e-01, -4.0615e-02,  4.9317e-02,  ..., -7.7130e-02,
           2.9478e-02,  2.0198e-02],
         [ 1.7174e-01, -9.1448e-02,  8.4149e-02,  ..., -4.4859e-02,
           1.3747e-01,  1.4645e-02],
         ...,
         [ 1.4262e-01, -1.1172e-01,  4.6165e-03,  ..., -2.1227e-02,
           4.6345e-02, -1.8608e-03],
         [ 1.7788e-01, -1.4028e-01,  7.4593e-02,  ..., -5.4616e-02,
          -2.0135e-02,  2.1813e-02],
         [ 1.7026e-01, -6.7175e-02,  3.2074e-02,  ..., -3.4447e-02,
           6.4736e-02, -4.2701e-03]],

        [[ 2.0407e-01, -1.6789e-01,  7.9332e-02,  ..., -1.0373e-01,
           9.6798e-02,  1.6005e-02],
         [ 1.9746e-01, -1.7385e-01,  2.7788e-02,  ..., -2.9735e-02,
           6.0901e-02,  2.7652e-02],
         [ 1.7308e-01, -1.1885e-01,  7.9013e-02,  ..., -1.1837e-01,
           1.9519e-02, -4.0333e-02],
         ...,
         [ 1.8115e-01, -7

# Grouped-Query Attention

In [35]:
class gqa(nn.Module):
    def __init__(self, hidden_dim, n_heads, n_kv):
        super().__init__()
        self.n_kv = n_kv
        self.hidden_dim = hidden_dim
        self.n_heads = n_heads
        assert n_heads % n_kv == 0

        self.head_dim = int(hidden_dim / n_heads)

        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, self.n_kv * self.head_dim)
        self.Wv = nn.Linear(hidden_dim, self.n_kv * self.head_dim)
        self.Wo = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(batch_size, seq_len, self.n_heads, -1).transpose(1,2)
        K = K.view(batch_size, seq_len, self.n_kv, -1).transpose(1,2)
        V = V.view(batch_size, seq_len, self.n_kv, -1).transpose(1,2)

        group_size = self.n_heads // self.n_kv
        K = K.repeat_interleave(group_size, dim = 1)
        V = V.repeat_interleave(group_size, dim = 1)
        
        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        weights = torch.softmax(scores, dim = -1)
        output = weights @ V
        output = output.transpose(1, 2).reshape(batch_size, seq_len, hidden_dim)
        output = self.Wo(output)

        return output

In [38]:
model = gqa(128, 8, 2)
x = torch.randn(4, 20, 128)
model(x)

tensor([[[-0.0123, -0.0514, -0.0520,  ...,  0.0640,  0.0759,  0.0084],
         [-0.0369, -0.0612, -0.0805,  ...,  0.0502,  0.0626,  0.0269],
         [ 0.0517, -0.1060, -0.0445,  ...,  0.0848,  0.0412,  0.0351],
         ...,
         [ 0.0171, -0.0674, -0.0530,  ...,  0.0753,  0.0767,  0.0357],
         [ 0.0455, -0.0817, -0.0918,  ...,  0.0652,  0.0223,  0.0276],
         [-0.0008, -0.0770, -0.0756,  ...,  0.0168,  0.0678,  0.0614]],

        [[ 0.0831, -0.0031,  0.1137,  ...,  0.0036, -0.0891, -0.0871],
         [ 0.1123,  0.0031,  0.1236,  ...,  0.0154, -0.0769, -0.0279],
         [ 0.1153,  0.0302,  0.1684,  ..., -0.0048, -0.1302, -0.0316],
         ...,
         [ 0.0622, -0.0250,  0.0709,  ...,  0.0396, -0.0988, -0.0614],
         [ 0.0910, -0.0061,  0.1046,  ...,  0.0109, -0.0914, -0.0364],
         [ 0.0579,  0.0333,  0.0974,  ..., -0.0103, -0.1077, -0.0296]],

        [[ 0.0701,  0.1073,  0.0091,  ...,  0.1027, -0.0585,  0.0174],
         [ 0.0912,  0.0872, -0.0230,  ...,  0